# Rosetta: Seamless Python-R Bioinformatics Workflow

Welcome to **Rosetta**, a framework designed to bridge the gap between Python and R/Bioconductor. 

### Core Philosophy
1. **Rosetta calls R — it doesn't reimplement it.** All statistics run in original, validated R packages.
2. **Pandas in, pandas out.** No complex R objects leak into your Python workflow.
3. **`.report()` everything.** Results are immediately interpretable without manual inspection.
4. **Show your work.** Built-in code generation lets you verify every underlying R call.


## Setup & Installation

Make sure you have installed `rosetta-bioc` and have a working R environment (R 4.0+) with required Bioconductor packages installed.

```bash
pip install rosetta-bioc
```


In [ ]:
import pandas as pd
import numpy as np
import rosetta as rb

print(f"Rosetta version: {rb.__version__}")


## Step 1: Data Preparation

Here we prepare standard count matrices and sample metadata (modeled after standard RNA-seq / airway datasets) to test our analysis pipeline.


In [ ]:
# Simulate an airway-style counts matrix (genes x samples)
np.random.seed(42)
genes = [f"ENSG00000{i}" for i in range(1000)]
samples = ["control_1", "control_2", "control_3", "treated_1", "treated_2", "treated_3"]

counts_df = pd.DataFrame(
    np.random.negative_binomial(10, 0.2, size=(len(genes), len(samples))),
    index=genes,
    columns=samples
)

metadata_df = pd.DataFrame(
    {"condition": ["control"] * 3 + ["treated"] * 3},
    index=samples
)

print("Counts shape:", counts_df.shape)
print(metadata_df)


## Step 2: Differential Expression Analysis (DESeq2)

Rosetta provides a **Three-Tier API** depending on how much control you need:
- **Tier 1 (Quick):** One-liner functions like `quick_deseq2()` for fast notebook analysis.
- **Tier 2 (Class-based):** Stateful classes (e.g., `Seurat`, `Phyloseq`) for chainable workflows.
- **Tier 3 (Functional):** Explicit step-by-step control (e.g., `run_deseq2()` + `get_results()`).


In [ ]:
# --- Tier 1: Quick API ---
# One call to fit and extract results
quick_results = rb.quick_deseq2(counts_df, metadata_df, design="~ condition", alpha=0.05)
quick_results.report()


In [ ]:
# --- Tier 2: Class-Based / Explicit Steps ---
model = rb.DESeq2(counts_df, metadata_df, design="~ condition")
model.run_deseq()

res = model.get_results(alpha=0.05)
res.report()


In [ ]:
# --- Tier 3: R Escape Hatch ---
model = rb.DESeq2(counts_df, metadata_df, design="~ condition")
model.run_deseq()

# Access the underlying rpy2 DESeqDataSet directly.
dds_r = model.r_obj

# Run arbitrary R code with the fitted object available as `obj`.
result_names = model.run_r_script("resultsNames(obj)")
print(list(result_names))

## Step 3: Advanced Features — Codegen

Don't trust a black box? Enable `codegen` to inspect the exact R commands executed behind the scenes.


In [ ]:
# --- Codegen with the DESeq2 class API ---
rb.codegen.enable()

try:
    model = rb.DESeq2(counts_df, metadata_df, design="~ condition")
    model.run_deseq()

    # Codegen currently records the DESeqDataSet creation and DESeq() R calls.
    print("Generated R Code:")
    print(rb.codegen.last())

    # Retrieve and summarize the results; the results() call is also recorded.
    deseq_results = model.get_results(alpha=0.05)
    deseq_results.report()
finally:
    rb.codegen.disable()


## Step 4: Single-Cell & Microbiome Analysis (Seurat & Phyloseq)

Rosetta also handles single-cell transcriptomics and microbiome profiles using class-based or quick interfaces.


In [ ]:
# --- Seurat Quick API Demo: genes × cells ---
np.random.seed(42)
sc_counts = pd.DataFrame(
    np.random.poisson(lam=2, size=(500, 50)),
    index=[f"gene_{i}" for i in range(500)],
    columns=[f"cell_{i}" for i in range(50)],
)

try:
    seurat_res = rb.quick_seurat(
        sc_counts,
        n_variable_features=500,
        n_pcs=2,
        resolution=0.5,
    )
    seurat_res.report()
except Exception as e:
    print("Seurat note:", e)

# --- Phyloseq Quick API Demo: taxa × samples ---
try:
    phy_res = rb.quick_phyloseq(
        otu_table=counts_df,
        sample_data=metadata_df,
        measures=["Shannon"],
    )
    phy_res.report()
except Exception as e:
    print("Phyloseq note:", e)


## Step 5: Downstream Enrichment Analysis

Connect your significant findings directly to functional annotations using `enrich_go` and `enrich_kegg`.


In [ ]:
# Example Entrez IDs for enrichment testing
sig_entrez = ["7157", "672", "675", "1956", "3845", "4609", "5290", "5728"]

try:
    kegg_res = rb.enrich_kegg(sig_entrez, organism="hsa")
    kegg_res.report()
except Exception as e:
    print("Enrichment note (requires clusterProfiler and KEGG access):", e)
